# Route A: a synchronous host-backed filesystem in the Pyodide worker

Issue #1780 wants Python in `python_exec` to see the person's real files (`open`, `os.walk`, `import`) through
`wmlhost`. Pyodide's filesystem is synchronous, the host is asynchronous. Route A blocks the Python worker on
`Atomics.wait` in slices of 50 ms, checking the interrupt flag between slices, while the request travels worker, offscreen
document, service worker, `wmlhost` and back. Route B rebuilds Pyodide with JSPI so a filesystem call can suspend instead.

Two questions, answered from measurements:

1. **Is route A's Stop latency and cost good enough, or is route B needed?** Stop latency: how long after Stop is pressed
   does a cell blocked in the filesystem raise `KeyboardInterrupt`, and does the interrupt leave the next cell clean.
   Cost: what one operation, one `os.walk` and one `import` cost, sandbox side and across the real message hops.
2. **What size cap does a whole-file fetch need?** `open().read()` on a mounted file fetches the whole file into the
   worker; where does it stop working, and does reading in chunks avoid it.

The sandbox side was measured in Node (`measure-node.mjs`, real Pyodide, a prototype filesystem type in a
`worker_threads` Worker, the host in the main thread serving requests from the real disk). The hops were measured in a
real Chromium with the built extension and a real `wmlhost` (`measure-hops.mjs`). Neither run contains the other, so
the end-to-end cost of a walk or an import is an **estimate composed from the two** (the last section), not a
measurement. The figures are one machine (see the first table); large files are sparse, so they time memory copies, not disk.

In [1]:
import json
import numpy as np
import pandas as pd
import wmlnb

HERE = wmlnb.ROOT / "notebooks" / "pyodide-mount"
DATA = HERE / "data"
FILES = sorted(p.name for p in DATA.glob("*.json"))
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 80)

def load(name):
    return json.loads((DATA / name).read_text())

D = {name[:-5]: load(name) for name in FILES}
MS = lambda s, k="median": s[k] if s else None

print(pd.DataFrame([{"file": f"data/{f}", "sha256": wmlnb.sha256_file(DATA / f)[:16]} for f in FILES]).to_string(index=False))
m, h = D["node-stop"]["machine"], D["hops-chain"]["machine"]
print()
print(f"sandbox side: {m['cpu']}, {m['cores']} cores, {m['memGB']} GB, node {m['node']}, {m['pyodide']}, {m['date']}")
print(f"hops: {h['cpu']}, chromium {h['chromium']}, node {h['node']}, quick={h['quick']}, host: wmlhost {D['hops-sw-host']['helloInfo']['version']} ({D['hops-sw-host']['helloInfo']['sandbox']['kind']} sandbox)")
print()
for c in D["node-stop"]["caveats"]:
    print("-", c)
print("hop failures recorded:", D["hops-failures"]["failures"] or "none")

                        file           sha256
        data/hops-chain.json f4c19250100b8761
     data/hops-failures.json 5f305323fcc20481
      data/hops-page-sw.json 65e973e540275982
data/hops-small-batched.json 374ec356d36f590f
      data/hops-sw-host.json 78074aac8d3b0f3e
         data/hops-wake.json 96b1d0ffa0912562
  data/hops-worker-page.json 96773d1cc77809e5
         data/node-cost.json efaf796748e3c8c2
       data/node-import.json 53603d59c5638c7b
        data/node-large.json be1b5046c6c4e3fd
     data/node-lockdown.json 8eb174039ca2e59d
         data/node-stop.json 6e6045358fb59d96
         data/node-walk.json f2468f7c799e7252

sandbox side: Apple M5 Pro, 15 cores, 24 GB, node v26.12.1, pyodide-core 314.0.3 (Python 3.14.0, restricted-mode fork build), 2026-11-13T22:35:23.695Z
hops: Apple M5 Pro, chromium 151.0.7922.34, node v26.12.1, quick=False, host: wmlhost 0.7.0 (seatbelt sandbox)

- The worker is a node:worker_threads Worker and the host a Node main thread: the channel is

## 1. Stop latency

A cell blocks inside a filesystem call (the host never answers); the stopper sets the interrupt flag, and in the
`notify` rows also wakes the waiter with `Atomics.notify`. `stopToHandler` is the time from the flag to Python's
`KeyboardInterrupt` handler running. A pure slice waits for the end of the current slice, so the latency is spread over about one
slice (the `max` column shows how far past 50 ms it goes); a notify cuts it to a wake-up. `flag cleared` is the interrupt word after the
handler (0 means the next cell is not re-interrupted), `next cell ok` is the count of repetitions where the following
cell ran normally, and `finally ops` counts filesystem operations a `finally:` block completed after the interrupt.

In [2]:
def stop_table(key):
    s = D[key]["stop"]
    rows = []
    for r in s["rows"]:
        rows.append({"blocked operation": r["name"], "notify": r["notify"], "reps": r["reps"],
                     "outcomes": json.dumps(r["outcomes"]),
                     "to handler median ms": MS(r["stopToHandlerMs"]), "p90": MS(r["stopToHandlerMs"], "p90"),
                     "max": MS(r["stopToHandlerMs"], "max"),
                     "flag cleared": json.dumps(r["interruptWordAfterHandler"]),
                     "next cell ok": r["nextCellNormal"], "finally ops": r["finallyOpsCompleted"],
                     "failures": len(r["failures"])})
    df = pd.DataFrame(rows)
    return s["sliceMs"], df

for key, title in [("node-stop", "no lockdown"), ("node-lockdown", "lockdownReadonly")]:
    slice_ms, df = stop_table(key)
    print(f"--- {title}; slice {slice_ms} ms ---")
    print(df.to_string(index=False))
    print()
late = D["node-stop"]["late"]; late_l = D["node-lockdown"]["late"]
print("a Stop that arrives after the call returned (no lockdown):", {k: late[k] for k in ("reps", "ok")}, "bad:", late["bad"])
print("a Stop that arrives after the call returned (lockdown):   ", {k: late_l[k] for k in ("reps", "ok")}, "bad:", late_l["bad"])

--- no lockdown; slice 50 ms ---
                                                                                           blocked operation  notify  reps                  outcomes  to handler median ms    p90    max flag cleared  next cell ok  finally ops  failures
                                                                  open().read(), blocked at the first lookup   False    25 {"KeyboardInterrupt": 25}                32.695 53.821 60.142    {"0": 25}            25            0         0
                                                     open().read(), lookup answered, whole-file read blocked   False    25 {"KeyboardInterrupt": 25}                25.908 56.078 60.115    {"0": 25}            25            0         0
                                                       open().read() of 20 MB, blocked on the 3rd 4 MB chunk   False    25 {"KeyboardInterrupt": 25}                36.784 52.489 60.096    {"0": 25}            25            0         0
                           

The aggregate below takes the worst row of each group: the `notify=False` rows are the pure slice, the `notify=True` rows
the same stop with a wake-up. Whether the slice's worst case is acceptable as a Stop is a judgement for the spec that cites this.

In [3]:
s, sl = stop_table("node-stop")[1], stop_table("node-lockdown")[1]
s["lockdown"], sl["lockdown"] = False, True
both = pd.concat([s, sl])
agg = both.groupby(["lockdown", "notify"]).agg(rows=("reps", "size"), reps=("reps", "sum"),
        worst_median_ms=("to handler median ms", "max"), worst_p90_ms=("p90", "max"), worst_max_ms=("max", "max"),
        next_cell_ok=("next cell ok", "sum"), finally_ops=("finally ops", "sum"), failures=("failures", "sum"))
agg

rows  reps  worst_median_ms  worst_p90_ms  worst_max_ms  next_cell_ok  finally_ops  failures
lockdown notify                                                                                              
False    False      6   150           36.784        57.483        60.142           150           25         0
         True       6   150            0.446         0.799         0.969           150           25         0
True     False      6   150           39.950        56.042        60.546           150           25         0
         True       6   150            0.375         0.646         0.983           150           25         0

## 2. Per-operation cost, sandbox side

One operation on the mount against the same operation on the in-memory filesystem (MEMFS), in Node, so the channel is one
`postMessage` plus a `SharedArrayBuffer` and the host answers in the same event-loop turn: this is the **floor** the
sandbox adds, before any hop. `rawSabRoundTrip` is the bare wait-and-wake with no filesystem. `requests` is what each
operation sent to the host. The last block is the same operation with a host that takes 1 ms or 5 ms to answer, which is
what a slow hop would look like to Python.

In [4]:
def cost_tables(key):
    c = D[key]["cost"]
    rows = [{"operation": k, "n": v["n"], "median ms": v["median"], "p90 ms": v["p90"], "max ms": v["max"]} for k, v in c["perOpMs"].items()]
    df = pd.DataFrame(rows)
    return c, df

for key, title in [("node-cost", "no lockdown"), ("node-lockdown", "lockdownReadonly")]:
    c, df = cost_tables(key)
    print(f"--- {title} ---")
    print(df.to_string(index=False))
    print("raw SAB round trip ms:", {k: c["rawSabRoundTripMs"][k] for k in ("n", "median", "p90", "max")})
    w = c["writeClose4KB"]
    print("write+close 4 KB:", {k: w["ms"][k] for k in ("n", "median", "p90", "max")}, "landed", w["landedBytes"], "bytes")
    print("requests per operation:", json.dumps(c["perOpRequests"]))
    print("with a slow host:", {k: {"median": v["median"], "p90": v["p90"], "max": v["max"]} for k, v in c["withHostDelay"].items()})
    print()

--- no lockdown ---
                                           operation   n  median ms  p90 ms  max ms
                      stat existing, mount (getattr) 500      0.017   0.025   0.186
                                stat existing, MEMFS 500      0.003   0.003   0.074
                   stat missing, mount (lookup miss) 500      0.025   0.027   0.101
                                 stat missing, MEMFS 500      0.003   0.003   0.007
                    open+read+close 4 KB text, mount 500      0.052   0.058   0.489
                    open+read+close 4 KB text, MEMFS 500      0.008   0.009   0.086
                           listdir 40 entries, mount 500      0.054   0.063   0.808
                           listdir 40 entries, MEMFS 500      0.008   0.012   0.143
scandir 40 entries + is_dir, mount, no listing cache 100      0.062   0.075   0.162
                  scandir 40 entries + is_dir, MEMFS 500      0.012   0.013   0.042
raw SAB round trip ms: {'n': 2000, 'median': 0.008, 'p90

In [5]:
c = D["node-cost"]["cost"]["perOpMs"]
pairs = {}
for k, v in c.items():
    if ", mount" in k:
        base = k.split(", mount")[0]
        other = next((o for o in c if o.startswith(base + ", MEMFS")), None)
        if other: pairs[base] = (v["median"], c[other]["median"])
pd.DataFrame([{"operation": b, "mount median ms": a, "MEMFS median ms": m_, "mount / MEMFS": round(a / m_, 1)} for b, (a, m_) in pairs.items()])

,operation,mount median ms,MEMFS median ms,mount / MEMFS
0,stat existing,0.017,0.003,5.7
1,stat missing,0.025,0.003,8.3
2,open+read+close 4 KB text,0.052,0.008,6.5
3,listdir 40 entries,0.054,0.008,6.8
4,scandir 40 entries + is_dir,0.062,0.012,5.2


## 3. Each hop's cost, in Chromium

The probe extension stands a page where the offscreen document stands, a dedicated worker where the Python worker
stands, and relays through the real service worker to a real `wmlhost`. Times are milliseconds. A single sample is
quantised to 0.1 ms by the browser's clock, so the medians of a one-sample timing read 0 or 0.1; `small-batched` times 20
back-to-back round trips and divides, which is the figure to trust below 0.1 ms.

In [6]:
def row(label, s, extra=None):
    d = {"hop": label, "n": s["n"], "median ms": s["median"], "p90": s["p90"], "max": s["max"], "min": s["min"]}
    d.update(extra or {})
    return d

rows = []
rows += [row("worker <-> page, " + r["kind"] + " " + r["label"], r) for r in D["hops-worker-page"]["rows"]]
rows += [row(f"page -> service worker, {r['dir']} {r['label']}", r) for r in D["hops-page-sw"]["rows"]]
rows += [row("batched mean, " + k, v) for k, v in D["hops-small-batched"].items() if isinstance(v, dict) and "median" in v]
sw = D["hops-sw-host"]
rows += [row("service worker <-> wmlhost, hello", sw["hello"]), row("service worker <-> wmlhost, hello batched", sw["helloBatched"])]
wk = D["hops-wake"]
rows += [row("worker <-> worker ping-pong (batched round trip)", wk["pingpong"]["roundTripMs"]),
         row("page notify -> parked worker returns (one way)", wk["parkedWake"]["oneWayMs"])]
print(D["hops-small-batched"]["note"])
print(D["hops-page-sw"]["note"])
print(D["hops-worker-page"]["note"])
print(wk["parkedWake"]["note"])
print(pd.DataFrame(rows).to_string(index=False))

mean per round trip of 20 back-to-back round trips (one sample alone is quantised to 0.1 ms)
chrome.runtime.sendMessage to a probe listener evaluated into the service worker; up = payload in request, down = payload in reply; base64 characters
round trip page->worker->page of the same payload both ways; extension page, not the offscreen document
page notify to the parked worker's return from Atomics.wait, on performance.timeOrigin+now (clock grain may be 0.1 ms)
                                             hop  n  median ms       p90       max       min
                        worker <-> page, str 1KB 30      0.000  0.100000  0.100000  0.000000
                       worker <-> page, str 64KB 30      0.100  0.200000  0.300000  0.000000
                        worker <-> page, str 1MB 30      0.500  0.600000  2.400000  0.300000
                        worker <-> page, str 8MB 20      3.000  3.400000  3.500000  2.600000
                         worker <-> page, ab 1KB 30      0.000  0.100

In [7]:
rows = []
for size, r in sw["reads"].items():
    rows.append({"file": size, "bytes": r["fileBytes"], "chunk": r["chunk"], "sequential MB/s": r["seqMBps"]["median"],
                 "sequential ms": r["seqMs"]["median"], "4-deep pipeline MB/s": r["pipe4MBps"]["median"],
                 "per chunk ms": r["perChunkMs"]["median"], "bytes ok": r["bytesOk"]})
print("service worker <-> wmlhost, reading held output in chunks")
print(pd.DataFrame(rows).to_string(index=False))
print()
rows = [{"output": size, "spawn ms": r["spawnMs"], "start answer ms": r["startAnswerMs"], "wall ms": r["wallMs"], "stdout bytes": r["stdout"]["size"], "error": r["error"]} for size, r in sw["spawn"].items()]
print(pd.DataFrame(rows).to_string(index=False))
print("host:", sw["helloInfo"]["host"], sw["helloInfo"]["version"], "sandbox", sw["helloInfo"]["sandbox"]["kind"], "; configured sandboxed:", sw["configured"]["sandboxed"])

service worker <-> wmlhost, reading held output in chunks
file     bytes  chunk  sequential MB/s  sequential ms  4-deep pipeline MB/s  per chunk ms  bytes ok
  f1   1048576 524288       208.333325            4.9            208.333325           2.4      True
 f16  16777216 524288       201.005025           79.8            280.210158           2.5      True
f100 104857600 524288       196.232339          513.7            283.446712           2.5      True

output  spawn ms  start answer ms  wall ms  stdout bytes error
    f1       0.2              1.2        6       1048576  None
   f16       0.2              0.6       12      16777216  None
  f100       0.1              0.7       47     104857600  None
host: window-ml-host 0.7.0 sandbox seatbelt ; configured sandboxed: True


## 4. The end-to-end chain

The Python worker's part is played by a worker that posts a request and blocks on `Atomics.wait` in 50 ms slices; the page
relays with `sendMessage`, the service worker relays to `wmlhost`, and the reply is written into the SAB. `hello` is the
smallest request (the **small-request round trip** the estimate below uses); files are read in 512 KiB chunks one at a
time and base64-decoded in the worker. `slices` is how many 50 ms slices a hello needed (1 means the reply landed
inside the first slice).

In [8]:
ch = D["hops-chain"]
print(ch["note"]); print()
print(pd.DataFrame([row("hello", ch["hello"], {"slices median": ch["hello"]["slices"]["median"], "slices max": ch["hello"]["slices"]["max"]}),
                    row("hello, batched mean", ch["helloBatched"])]).to_string(index=False))
print()
rows = []
for size, r in ch["files"].items():
    rows.append({"file": size, "bytes": r["fileBytes"], "n": r["ms"]["n"], "median ms": r["ms"]["median"], "p90 ms": r["ms"]["p90"],
                 "MB/s median": r["MBps"]["median"], "MB/s min": r["MBps"]["min"], "per chunk rpc ms": r["perChunkRpcMs"]["median"],
                 "per chunk decode ms": r["perChunkDecodeMs"]["median"], "bytes ok": r["bytesOk"]})
print(pd.DataFrame(rows).to_string(index=False))

worker posts a request and blocks on Atomics.wait in 50 ms slices; page relays with sendMessage; service worker relays to wmlhost; reply is written into the SAB as JSON; files are read in 512 KiB chunks one at a time, base64 decoded in the worker

                hop  n  median ms  p90  max  min  slices median  slices max
              hello 28      0.300 0.30 0.40 0.20            1.0         1.0
hello, batched mean 25      0.255 0.29 0.39 0.25            NaN         NaN

file     bytes  n  median ms  p90 ms  MB/s median   MB/s min  per chunk rpc ms  per chunk decode ms  bytes ok
  f1   1048576 30        9.7    10.7   103.092786  72.992700               4.3                  0.6      True
 f16  16777216  6      154.8   156.6   103.492885 102.171137               4.2                  0.6      True
f100 104857600  3      974.6   977.1   102.606197 102.343670               4.2                  0.6      True


## 5. `os.walk` by cache mode

A walk of the real workspace of this checkout, with two directories hidden or none, three cache modes (no listing cache; a
listing cache; a listing cache plus a directory cache), against the host's own count of files and directories (`python saw` must equal
`host truth`). `first walk` is the cell's first `os.walk`, `second walk` the same walk again in the same cell. Without a
cache the walk costs one `stat` per entry plus a `readdir` per directory; a listing that carries each entry's type removes
the per-entry `stat`.

In [9]:
rows = []
for r in D["node-walk"]["walk"]["rows"]:
    mode = "listing+dir cache" if r["dirCache"] else "listing cache" if r["listingCache"] else "no cache"
    rows.append({"hidden": r["hidden"], "cache": mode, "files": r["hostTruth"]["files"], "dirs": r["hostTruth"]["dirs"],
                 "python saw == host": r["pythonSaw"] == r["hostTruth"], "reps": r["reps"],
                 "first walk ms": MS(r["firstWalkMs"]), "first p90": MS(r["firstWalkMs"], "p90"),
                 "first requests": MS(r["firstWalkRequests"]), "by op": json.dumps(r["firstWalkRequestsByOp"]),
                 "second walk ms": MS(r["secondWalkSameCellMs"]), "second requests": MS(r["secondWalkRequests"]),
                 "us/entry first": r["usPerEntryFirstWalk"], "error": r["error"]})
walk_df = pd.DataFrame(rows)
print("workspace:", D["node-walk"]["walk"]["workspace"])
print(walk_df.to_string(index=False))

workspace: /Users/sb/git/window-ml-async-ra
           hidden             cache  files  dirs  python saw == host  reps  first walk ms  first p90  first requests                                            by op  second walk ms  second requests  us/entry first error
        (nothing)          no cache  21498  1558                True    20        352.671    373.517           26238 {"readdir": 1559, "stat": 24645, "readlink": 34}         103.205             3182          15.296  None
        (nothing)     listing cache  21498  1558                True    20        131.273    138.519            1639    {"readdir": 1559, "readlink": 34, "stat": 46}         127.791             1593           5.694  None
        (nothing) listing+dir cache  21498  1558                True    20        132.806    141.087            1639    {"readdir": 1559, "readlink": 34, "stat": 46}          36.660               34           5.760  None
node_modules,.git          no cache   2439   185                True    

## 6. `import` by variant

Importing a package from the mount, two cases (a synthetic package of 50 modules and a real pure-Python package, pip's
vendored `rich`), by whether Python may write a `.pyc` beside the source. A `.pyc` written to the mount costs a `put`
and a `rename` per module on the first import; a second process then reads them. `channel requests` is the number of host
round trips one import makes; MEMFS rows make none.

In [10]:
rows = []
for r in D["node-import"]["import"]["rows"]:
    rows.append({"case": r["case"].split(",")[0][:40], "variant": r["variant"], "files": r["files"], "reps": r["reps"],
                 "import median ms": MS(r["importMs"]), "p90": MS(r["importMs"], "p90"), "modules": r["modulesImported"],
                 "channel requests": MS(r["channelRequests"]), "by op": json.dumps(r["requestsByOp"]), "error": r["error"]})
import_df = pd.DataFrame(rows)
print(import_df.to_string(index=False))

                                   case                                     variant  files  reps  import median ms    p90  modules  channel requests                                                                        by op error
                   synthetic 50 modules   MEMFS, first import (the .pyc is written)     51    20            25.979 30.217       51                 0                                                                           {}  None
                   synthetic 50 modules          MEMFS, later import reads the .pyc     51    19            12.557 14.145       51                 0                                                                           {}  None
                   synthetic 50 modules          mount read-only (no .pyc possible)     51    20            30.562 33.006       51               211                                      {"stat": 158, "readdir": 2, "read": 51}  None
                   synthetic 50 modules                  mount, dont_wri

## 7. A large file opened whole, and where `f.read()` breaks

`open_fetch` is `open()` on the mount (which fetches the whole file into the worker in the prototype); `read_all` is
`f.read()`; `read_1MiB_loop` reads the same file in 1 MiB `read(n)` calls. The files are sparse, so the times are memory
copies. The sizes run past 2 GiB (the largest `bytes` object) to 4 GiB. The `cap` row tests a host-side fetch cap
(`maxFetch`) smaller than the file.

In [11]:
L = D["node-large"]["large"]
rows, caps = [], []
for r in L["rows"]:
    if "cap" in r:
        caps.append(r); continue
    ra, lp = r["read_all"]["value"], r["read_1MiB_loop"]["value"]
    rows.append({"size MB": r["sizeMB"], "rep": r["rep"], "open+fetch ms": r["open_fetch"]["hostMs"],
                 "f.read()": ra.get("exc") or "ok", "f.read() bytes": ra.get("len"), "f.read() ms": round(ra["ms"], 1),
                 "read(1 MiB) loop": lp.get("exc") or "ok", "loop bytes": lp.get("len"), "loop ms": round(lp["ms"], 1),
                 "requests": json.dumps(r["requests"]), "free GB before": r["freeGBBefore"]})
large_df = pd.DataFrame(rows)
print(large_df.to_string(index=False))
print()
for c in caps:
    print("cap:", c["cap"], "->", c["result"].split(":")[0] + ":" + c["result"].split(":")[1][:40], c["requests"])
print()
print("chunk size sweep, a 100 MB file opened")
print(pd.DataFrame([{"chunk bytes": s["chunkBytes"], "reps": s["reps"], "open median ms": MS(s["fetch100MBms"]), "p90": MS(s["fetch100MBms"], "p90"),
               "requests per open": s["requestsPerOpen"], "MB/s median": s["MBperS_median"], "error": s["error"]} for s in L["chunkSweep"]]).to_string(index=False))

 size MB  rep  open+fetch ms      f.read()  f.read() bytes  f.read() ms read(1 MiB) loop  loop bytes  loop ms                  requests  free GB before
     100    0          11.25            ok     104857600.0          5.2               ok   104857600      1.5   {"stat": 1, "read": 25}           8.755
     100    1           9.70            ok     104857600.0          5.4               ok   104857600      1.5   {"stat": 1, "read": 25}           8.662
     100    2           9.83            ok     104857600.0          5.5               ok   104857600      1.5   {"stat": 1, "read": 25}           8.659
     100    3           9.73            ok     104857600.0          5.4               ok   104857600      1.5   {"stat": 1, "read": 25}           8.657
     100    4           9.67            ok     104857600.0          5.4               ok   104857600      1.5   {"stat": 1, "read": 25}           8.663
     512    0          67.52            ok     536870912.0         26.1               ok

In [12]:
ok = large_df[large_df["f.read()"] == "ok"]["size MB"]
bad = large_df[large_df["f.read()"] != "ok"]
loop_ok = large_df[large_df["read(1 MiB) loop"] == "ok"]["size MB"]
print(f"f.read() works up to {ok.max()} MB in this run")
print("f.read() fails:", {int(r["size MB"]): r["f.read()"] for _, r in bad.iterrows()})
print(f"the 1 MiB loop works up to {loop_ok.max()} MB (the largest size measured)")

f.read() works up to 2047 MB in this run
f.read() fails: {2048: 'OverflowError', 3072: 'OverflowError', 4096: 'MemoryError'}
the 1 MiB loop works up to 4096 MB (the largest size measured)


## 8. What a lockdown run can reach of the channel

Probes run in a cell under `lockdownReadonly` (the readonly default, the bridge to the browser's JavaScript refused) and
with the bridge lifted (a cell the person approved for the browser's JavaScript). A reach that shows a raw object (the
`SharedArrayBuffer` or the interrupt word) would let a cell forge a reply or clear the interrupt.

In [13]:
P = D["node-lockdown"]["reach"]["probes"]
names = list(P["bridge refused (the readonly default)"])
def short(v):
    s = json.dumps(v) if not isinstance(v, str) else v
    return s if len(s) <= 90 else s[:87] + "..."
print(pd.DataFrame([{"probe": n, "bridge refused": short(P["bridge refused (the readonly default)"][n]), "bridge lifted": short(P["bridge lifted"][n])} for n in names]).to_string(index=False))

                               probe                                                                             bridge refused                                                                              bridge lifted
          import js: globals present RAISED ImportError: js is refused in python_exec: a cell may not reach the browser's Ja...                                                           ["Atomics", "SharedArrayBuffer"]
  js.Atomics / new SharedArrayBuffer RAISED ImportError: js is refused in python_exec: a cell may not reach the browser's Ja...                                                       can build a NEW SharedArrayBuffer: 8
js.globalThis lowercase and __ names RAISED ImportError: js is refused in python_exec: a cell may not reach the browser's Ja... ["atob", "btoa", "clearInterval", "clearTimeout", "console", "crypto", "decodeURI", "de...
              Atomics on a fresh SAB RAISED ImportError: js is refused in python_exec: a cell may not reach the browser's Ja

## 9. Derived: the estimated product cost of `os.walk` and `import`

**This is an estimate composed from two measurements, not a measurement.** The sandbox-side time (Node, section 5 and 6) was
measured with a host that answers in the same event-loop turn, so it holds the Python and filesystem work and one cheap
hop per request. The chain's small-request round trip (section 4) was measured in Chromium through every real hop. The
estimate is `requests x chain round trip + Node sandbox-side time`. It counts each request's Node hop twice (about 8 us each, the raw
SAB round trip) and assumes requests are issued one at a time, as the sync filesystem must, and that `wmlhost` answers a
`stat` or `readdir` in the time it answers `hello`, which section 3 measured only for `hello`. Two round-trip figures are used because one sample is quantised to 0.1 ms:
the median of single `hello` round trips, and the median of the 20-round-trip batches.

In [14]:
RTT_SINGLE = D["hops-chain"]["hello"]["median"]
RTT_BATCH = D["hops-chain"]["helloBatched"]["median"]
print(f"chain small-request round trip: single-sample median {RTT_SINGLE:.3f} ms; batched median {RTT_BATCH:.3f} ms")

def est(req, node_ms):
    return pd.Series({"requests": req, "node ms": node_ms,
                      "est ms (single rtt)": round(req * RTT_SINGLE + node_ms, 1),
                      "est ms (batched rtt)": round(req * RTT_BATCH + node_ms, 1)})

w = []
for r in D["node-walk"]["walk"]["rows"]:
    mode = "listing+dir cache" if r["dirCache"] else "listing cache" if r["listingCache"] else "no cache"
    w.append(pd.concat([pd.Series({"what": f"os.walk, hidden: {r['hidden']}, {mode}"}), est(MS(r["firstWalkRequests"]), MS(r["firstWalkMs"]))]))
i = []
for r in D["node-import"]["import"]["rows"]:
    if r["variant"].startswith("MEMFS"): continue
    i.append(pd.concat([pd.Series({"what": f"import, {'real rich' if r['case'].startswith('real') else r['case']}, {r['variant']}"}), est(MS(r["channelRequests"]), MS(r["importMs"]))]))
est_df = pd.DataFrame(w + i).set_index("what")
est_df["node share"] = (est_df["node ms"] / est_df["est ms (batched rtt)"]).round(2)
print(est_df.to_string())

chain small-request round trip: single-sample median 0.300 ms; batched median 0.255 ms
                                                                           requests  node ms  est ms (single rtt)  est ms (batched rtt)  node share
what                                                                                                                                               
os.walk, hidden: (nothing), no cache                                        26238.0  352.671               8224.1                7043.4        0.05
os.walk, hidden: (nothing), listing cache                                    1639.0  131.273                623.0                 549.2        0.24
os.walk, hidden: (nothing), listing+dir cache                                1639.0  132.806                624.5                 550.8        0.24
os.walk, hidden: node_modules,.git, no cache                                 2998.0   39.369                938.8                 803.9        0.05
os.walk, hidden: node_mod

In [15]:
base = {}
for r in D["node-import"]["import"]["rows"]:
    if r["variant"].startswith("MEMFS, later"):
        base[r["case"]] = MS(r["importMs"])
imp = est_df[est_df.index.str.startswith("import")].copy()
imp["MEMFS later-import ms"] = [base[next(c for c in base if ("real rich" if c.startswith("real") else c) in n.split(", ", 1)[1])] for n in imp.index]
imp["est / MEMFS (batched)"] = (imp["est ms (batched rtt)"] / imp["MEMFS later-import ms"]).round(1)
print(imp[["requests", "est ms (batched rtt)", "MEMFS later-import ms", "est / MEMFS (batched)"]].to_string())

                                                                           requests  est ms (batched rtt)  MEMFS later-import ms  est / MEMFS (batched)
what                                                                                                                                                   
import, synthetic 50 modules, mount read-only (no .pyc possible)              211.0                  84.4                 12.557                    6.7
import, synthetic 50 modules, mount, dont_write_bytecode                      109.0                  54.1                 12.557                    4.3
import, synthetic 50 modules, mount read-write, first import writes .pyc      418.0                 149.2                 12.557                   11.9
import, synthetic 50 modules, mount read-write, second process reads .pyc     110.0                  43.4                 12.557                    3.5
import, real rich, mount read-only (no .pyc possible)                         149.0     

## Reading the results

This notebook keeps the figures; the ruling on route A against route B, and the size cap, belongs in the spec that
cites a section of it (README.md names the scripts that wrote the inputs).